In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell: Visualise top-activating images per concept — AFTER unlearning
#
# What this does:
#   1. Loads the unlearned probe weights  (binary_probe_finetuned_k<N>.pt)
#   2. Re-ranks all concepts by |new weight| → new top-N concepts
#   3. For each top concept: finds the images with highest activation for it
#   4. Saves one PNG grid per concept  →  outputs/Mouth_Slightly_Open/vis_unlearned/
#   5. Also saves updated top_concepts_unlearned.txt + all_concepts_ranked_unlearned.csv
# ══════════════════════════════════════════════════════════════════════════════

import os, csv
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from pathlib import Path
from torchvision import transforms

# ── 0. Paths — edit these to match your setup ────────────────────────────────
ATTRIBUTE          = "Attractive"
OUTPUT_DIR         = f"./unlearning/outputs/{ATTRIBUTE}"
UNLEARNED_CKPT     = f"./unlearning/outputs/Attractive/binary_probe_unlearned_k100_polpositive.pt"
CONCEPT_NAMES_CSV  = "./checkpoints/clip_RN50_concept_name.csv"   # idx,name,sim
SAE_ACTIVATIONS    = "./data/celeba_sae_activations/train"        # .pth tensor [N, 8192]
CELEBA_IMG_DIR     = "./data/celeba/img_align_celeba"             # folder of .jpg files
CELEBA_SPLIT_FILE  = "./data/celeba/list_eval_partition.csv"      # or Anno/list_eval_partition.txt

VIS_DIR            = os.path.join(OUTPUT_DIR, "vis_unlearned")
TOP_N_CONCEPTS     = 10     # how many top concepts to visualise
IMGS_PER_CONCEPT   = 9      # images shown per concept (rows × cols below)
GRID_ROWS, GRID_COLS = 3, 3
assert GRID_ROWS * GRID_COLS == IMGS_PER_CONCEPT

os.makedirs(VIS_DIR, exist_ok=True)

# ── 1. Load concept names ─────────────────────────────────────────────────────
concept_names = {}
if os.path.exists(CONCEPT_NAMES_CSV):
    with open(CONCEPT_NAMES_CSV, newline="") as f:
        for row in csv.reader(f):
            if len(row) >= 2:
                try:
                    concept_names[int(row[0])] = row[1].strip()
                except ValueError:
                    pass
print(f"✓ Loaded {len(concept_names)} concept names")

# ── 2. Load unlearned probe weights ──────────────────────────────────────────
ckpt    = torch.load(UNLEARNED_CKPT, map_location="cpu")
weights = ckpt["model_state"]["linear.weight"].squeeze()   # [n_concepts]
n_concepts = weights.shape[0]
print(f"✓ Unlearned weights loaded — shape: {weights.shape}")

# ── 3. Re-rank concepts by |weight| and write updated CSVs ───────────────────
sorted_indices = torch.argsort(weights.abs(), descending=True)

# top_concepts_unlearned.txt
top_txt_path = os.path.join(OUTPUT_DIR, "top_concepts_unlearned.txt")
with open(top_txt_path, "w") as f:
    for rank, idx in enumerate(sorted_indices[:TOP_N_CONCEPTS]):
        w    = weights[idx].item()
        name = concept_names.get(int(idx), f"concept_{int(idx)}")
        line = f"  {rank+1:2d}. {name:<35} (idx={int(idx)}) | Weight: {w:+.6f}"
        f.write(line + "\n")
        print(line)
print(f"\n✓ Saved → {top_txt_path}")

# all_concepts_ranked_unlearned.csv
ranked_csv_path = os.path.join(OUTPUT_DIR, "all_concepts_ranked_unlearned.csv")
with open(ranked_csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["rank", "concept_idx", "name", "weight"])
    for rank, idx in enumerate(sorted_indices):
        idx_int = int(idx)
        writer.writerow([rank + 1, idx_int,
                         concept_names.get(idx_int, ""),
                         f"{weights[idx].item():+.6f}"])
print(f"✓ Saved → {ranked_csv_path}")

# ── 4. Load SAE concept-strength activations ─────────────────────────────────
print(f"\n→ Loading concept strengths from: {SAE_ACTIVATIONS}")
all_concepts = torch.load(SAE_ACTIVATIONS, map_location="cpu")   # [N, n_concepts]
if all_concepts.ndim == 3 and all_concepts.shape[1] == 1:
    all_concepts = all_concepts.squeeze(1)
print(f"✓ Concept strengths shape: {all_concepts.shape}")
N = all_concepts.shape[0]

# ── 5. Build ordered list of CelebA train image filenames ────────────────────
# Reads list_eval_partition.csv (or .txt) to find partition==0 (train) images
# and preserves the same order as the saved activations.
img_files = []
partition_path = Path(CELEBA_SPLIT_FILE)
if partition_path.suffix == ".csv":
    with open(partition_path, newline="") as f:
        reader = csv.reader(f)
        next(reader, None)   # skip header if present
        for row in reader:
            if len(row) >= 2 and row[1].strip() == "0":
                img_files.append(row[0].strip())
else:   # plain txt: "000001.jpg 0"
    with open(partition_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2 and parts[1] == "0":
                img_files.append(parts[0])

assert len(img_files) == N, (
    f"Image list length ({len(img_files)}) ≠ activations length ({N}). "
    "Check CELEBA_SPLIT_FILE and SAE_ACTIVATIONS are aligned to the same split/order.")
print(f"✓ {len(img_files)} train images indexed")

# ── 6. CLIP un-normalisation (same as vis scripts in this repo) ───────────────
un_normalize = transforms.Normalize(
    mean=(-0.48145466/0.26862954, -0.4578275/0.26130258, -0.40821073/0.27577711),
    std =(1/0.26862954, 1/0.26130258, 1/0.27577711)
)
clip_preprocess = transforms.Compose([
    transforms.Resize(224, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize((0.48145466, 0.4578275, 0.40821073),
                         (0.26862954, 0.26130258, 0.27577711)),
])

def load_celeba_img(fname):
    """Load, preprocess then un-normalise a CelebA image → numpy [H,W,3] in [0,1]."""
    path = os.path.join(CELEBA_IMG_DIR, fname)
    img  = Image.open(path).convert("RGB")
    t    = clip_preprocess(img)           # [3,224,224] normalised
    t    = un_normalize(t)                # back to [0,1] range
    return t.permute(1, 2, 0).numpy().clip(0, 1)

# ── 7. For each top concept: find top-activating images + save PNG ────────────
top_concept_indices = sorted_indices[:TOP_N_CONCEPTS]

for rank, concept_idx in enumerate(top_concept_indices):
    c_idx  = int(concept_idx)
    w_val  = weights[concept_idx].item()
    c_name = concept_names.get(c_idx, f"concept_{c_idx}")

    # top images for this concept
    activations_for_concept = all_concepts[:, c_idx]          # [N]
    _, top_img_idxs = torch.topk(activations_for_concept, k=IMGS_PER_CONCEPT)
    top_img_idxs    = top_img_idxs.tolist()

    # build grid
    fig = plt.figure(figsize=(GRID_COLS * 2.5, GRID_ROWS * 2.5 + 0.8))
    fig.suptitle(
        f"Rank {rank+1}  |  \"{c_name}\"  (idx={c_idx})  |  weight={w_val:+.4f}",
        fontsize=11, fontweight="bold", y=0.98
    )
    gs  = gridspec.GridSpec(GRID_ROWS, GRID_COLS, figure=fig,
                            hspace=0.05, wspace=0.05)

    for pos, img_idx in enumerate(top_img_idxs):
        ax  = fig.add_subplot(gs[pos // GRID_COLS, pos % GRID_COLS])
        img = load_celeba_img(img_files[img_idx])
        ax.imshow(img)
        act_val = activations_for_concept[img_idx].item()
        ax.set_title(f"act={act_val:.2f}", fontsize=7, pad=2)
        ax.axis("off")

    png_path = os.path.join(VIS_DIR, f"rank{rank+1:02d}_concept{c_idx}_{c_name.replace(' ','_')}.png")
    fig.savefig(png_path, dpi=120, bbox_inches="tight")
    plt.close(fig)
    print(f"  [{rank+1:2d}/{TOP_N_CONCEPTS}] Saved → {png_path}")

print(f"\n✓ All {TOP_N_CONCEPTS} concept PNGs saved to: {VIS_DIR}")

FileNotFoundError: [Errno 2] No such file or directory: './unlearning/outputs/Attractive/binary_probe_unlearned_k100_polpositive.pt'